# Visualizations for alignment matrices

In [ ]:
"""
importing libraries
"""
from datetime import datetime
import pandas as pd
from pathlib import Path
import pickle as pl
import numpy as np
import seaborn as sns  
import matplotlib.pyplot as plt
import json
from datasets import load_dataset
import re

### Heatmaps for mean alignment scores

In [ ]:
# =========================================================
# DATASET LOADING
# =========================================================
def load_train_dataset(dataset_name: str):
    """
    Load the training split of a supported HuggingFace dataset.

    Args:
        dataset_name (str): Dataset identifier (ag_news, snli, yelp_review, yelp_review_full).

    Returns:
        Dataset: Training split of the requested dataset.

    Raises:
        ValueError: If dataset_name is not supported.
    """
    if dataset_name == "ag_news":
        return load_dataset("ag_news", split="train")
    if dataset_name == "snli":
        return load_dataset("snli", split="train")
    if dataset_name in {"yelp_review", "yelp_review_full"}:
        return load_dataset("yelp_review_full", split="train")
    raise ValueError(f"Unsupported dataset_name: {dataset_name}")


def get_class_names(dataset_name: str, num_classes: int):
    """
    Return human-readable class labels for a given dataset.

    Args:
        dataset_name (str): Dataset identifier.
        num_classes (int): Number of classes (used as fallback).

    Returns:
        List[str]: List of class names.
    """
    class_names_map = {
        "ag_news": ["World", "Sports", "Business", "Sci/Tech"],
        "snli": ["Entailment", "Neutral", "Contradiction"],
        "yelp_review": ["1★", "2★", "3★", "4★", "5★"],
        "yelp_review_full": ["1★", "2★", "3★", "4★", "5★"],
    }
    return class_names_map.get(dataset_name, [f"C{i}" for i in range(num_classes)])


# =========================================================
# PARSING
# =========================================================
def extract_proportion_tuple(name: str):
    """
    Extract a tuple of floats from a string containing values in
    square brackets [] or parentheses ().

    Args:
        name (str): Input string (e.g., "exp_[0.2, 0.3, 0.5]").

    Returns:
        tuple[float] or None: Extracted values as a tuple, or None if no match.
    """
    m = re.search(r"(\[[^\]]+\]|\([^)]*\))", name)
    if not m:
        return None
    raw = m.group(1)[1:-1]
    return tuple(float(x.strip()) for x in raw.split(",") if x.strip())


def parse_align_dir_name(name: str, dataset_name: str):
    """
    Parse directory name to extract dataset, method, and proportion tuple.

    Args:
        name (str): Directory name containing dataset, method, and proportions.
        dataset_name (str): Expected dataset prefix.

    Returns:
        dict or None: Parsed components {"dataset", "method", "proportion"},
        or None if format is invalid.
    """
    prop = extract_proportion_tuple(name)
    if prop is None:
        return None

    # Remove the proportion part
    prefix = re.sub(r"(\[[^\]]+\]|\([^)]*\))", "", name).rstrip("_")

    expected_prefix = f"{dataset_name}_"
    if not prefix.startswith(expected_prefix):
        return None

    method = prefix[len(expected_prefix):]
    if not method:
        return None

    return {
        "dataset": dataset_name,
        "method": method,
        "proportion": prop,
    }


def parse_datainfo_dir_name(name: str):
    """
    Parse directory name to extract dataset and proportion tuple.

    Args:
        name (str): Directory name (e.g., "ag_news_(...)").

    Returns:
        dict or None: {"dataset", "proportion"} or None if parsing fails.
    """
    prop = extract_proportion_tuple(name)
    if prop is None:
        return None

    prefix = name.split("(")[0].rstrip("_")
    return {
        "dataset": prefix,
        "proportion": prop,
    }


# =========================================================
# MATCHING
# =========================================================
def collect_alignment_dirs(align_root: Path, dataset_name: str, method: str | None = None):
    """
    Collect alignment directories mapped by proportion tuples.

    Args:
        align_root (Path): Root directory containing alignment subdirs.
        dataset_name (str): Dataset prefix to filter directories.
        method (str | None): Optional method filter.

    Returns:
        dict: {proportion_tuple: Path} for matching directories.
    """
    out = {}
    for p in align_root.iterdir():
        if not p.is_dir():
            continue
        parsed = parse_align_dir_name(p.name, dataset_name)
        if parsed is None:
            continue
        if method is not None and parsed["method"] != method:
            continue
        out[parsed["proportion"]] = p
    return out


def collect_datainfo_dirs(datainfo_root: Path, dataset_name: str):
    """
    Collect datainfo directories mapped by proportion tuples.

    Args:
        datainfo_root (Path): Root directory containing datainfo subdirs.
        dataset_name (str): Dataset prefix to filter directories.

    Returns:
        dict: {proportion_tuple: Path} for matching directories.
    """
    out = {}
    for p in datainfo_root.iterdir():
        if not p.is_dir():
            continue
        parsed = parse_datainfo_dir_name(p.name)
        if parsed is None:
            continue
        if parsed["dataset"] != dataset_name:
            continue
        out[parsed["proportion"]] = p
    return out


def get_matched_pairs( align_root,datainfo_root,dataset_name,method,selected_proportions=None):
    """
    Match alignment and datainfo directories by common proportion tuples.

    Args:
        align_root (Path | str): Root of alignment directories.
        datainfo_root (Path | str): Root of datainfo directories.
        dataset_name (str): Dataset to filter.
        method (str): Alignment method to filter.
        selected_proportions (list | None): Optional subset of proportions.
        max_pairs (int | None): Optional limit on number of pairs.

    Returns:
        list[tuple]: [(proportion, align_path, datainfo_path), ...]
    """
    align_root = Path(align_root)
    datainfo_root = Path(datainfo_root)

    align_dirs = collect_alignment_dirs(align_root, dataset_name, method=method)
    datainfo_dirs = collect_datainfo_dirs(datainfo_root, dataset_name)

    common_props = sorted(set(align_dirs) & set(datainfo_dirs))

    if selected_proportions is not None:
        selected_set = {tuple(map(float, p)) for p in selected_proportions}
        common_props = [p for p in common_props if p in selected_set]

    return [(prop, align_dirs[prop], datainfo_dirs[prop]) for prop in common_props]


# =========================================================
# CORE COMPUTATION
# =========================================================
def compute_per_class_alignment(alignment_matrix: np.ndarray, labels: np.ndarray) -> np.ndarray:
    """
    Compute mean alignment scores per class for each pseudo-expert.

    Args:
        alignment_matrix (np.ndarray): Shape (N, K) with alignment scores.
        labels (np.ndarray): Shape (N,) with class labels.

    Returns:
        np.ndarray: Shape (K, C) where each entry is the mean alignment
        of expert k for class c (NaN if no samples for class).
    """
    if alignment_matrix.ndim != 2:
        raise ValueError(f"Expected 2D alignment_matrix, got shape {alignment_matrix.shape}")

    N, K = alignment_matrix.shape
    if N != len(labels):
        raise ValueError(
            f"Mismatch: alignment_matrix has {N} samples, but labels has {len(labels)} entries."
        )

    num_classes = int(labels.max()) + 1
    out = np.full((K, num_classes), np.nan, dtype=np.float32)

    for k in range(K):
        for c in range(num_classes):
            mask = labels == c
            if np.any(mask):
                out[k, c] = alignment_matrix[mask, k].mean()

    return out


def get_labels_from_datainfo(dataset, datainfo_json_path: Path):
    """
    Extract labels for selected indices from a dataset using a datainfo JSON.

    Args:
        dataset: Dataset object with "label" field.
        datainfo_json_path (Path): Path to JSON containing "indices_D".

    Returns:
        tuple: (labels array, dataset_info dict)
    """
    with open(datainfo_json_path, "r") as f:
        dataset_info = json.load(f)

    valid_indices = dataset_info["indices_D"]
    labels = np.array([dataset[idx]["label"] for idx in valid_indices], dtype=np.int64)
    return labels, dataset_info


# =========================================================
# FILE DISCOVERY
# =========================================================
def find_alignment_matrix_file(align_dir: Path, method="linear"):
    preferred = align_dir / f"alignment_matrix_{method}.npy"
    if preferred.exists():
        return preferred

    generic = sorted(align_dir.glob("alignment_matrix*.npy"))
    if generic:
        if len(generic) > 1:
            print(f"Warning: multiple alignment files in {align_dir}, using {generic[0].name}")
        return generic[0]

    raise FileNotFoundError(f"No alignment matrix file found in {align_dir}")


# =========================================================
# HEATMAP
# =========================================================
def save_alignment_heatmap( per_class_alignment: np.ndarray, class_proportions: np.ndarray, dataset_name: str, save_path: Path, title: str | None = None,):
    """
    Generate and save a heatmap of per-class alignment scores.

    Args:
        per_class_alignment (np.ndarray): Shape (K, C) alignment matrix.
        class_proportions (np.ndarray): Class distribution proportions.
        dataset_name (str): Dataset identifier for class labels.
        save_path (Path): Output path for the heatmap image.
        title (str | None): Optional plot title.
    """
    num_classes = per_class_alignment.shape[1]
    K = per_class_alignment.shape[0]

    class_names = get_class_names(dataset_name, num_classes)
    yticklabels = [
        f"{class_names[i]} ({100 * class_proportions[i]:.1f}%)"
        for i in range(num_classes)
    ]

    plt.figure(figsize=(12, 8))
    sns.heatmap(
        per_class_alignment.T,
        annot=True,
        fmt=".3f",
        cmap="RdYlGn",
        xticklabels=[f"θ_{i}" for i in range(K)],
        yticklabels=yticklabels,
        cbar_kws={"label": "Mean Alignment Score"},
        linewidths=0.5,
    )

    plt.xlabel("Pseudo-Expert Index", fontsize=12)
    plt.ylabel("Class", fontsize=12)
    plt.title(
        title or "Mean Alignment Score per Class across Pseudo-Experts",
        fontsize=14,
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight", format="png")
    plt.close()


# =========================================================
# GENERATE ONE
# =========================================================
def generate_one_per_class_alignment( align_dir: Path, datainfo_dir: Path, dataset, output_root: Path, dataset_name, method, 
output_filename="per_class_alignment.npy", 
metadata_filename="metadata.json",
heatmap_filename="per_class_alignment_heatmap.png"):
    """
    Generate, save, and visualize per-class alignment for one matched pair
    of alignment and datainfo directories.

    Args:
        align_dir (Path): Directory containing alignment matrix file.
        datainfo_dir (Path): Directory containing dataset info JSON.
        dataset: Dataset object used to fetch labels.
        output_root (Path): Root directory for saving outputs.
        dataset_name (str): Dataset identifier.
        method (str): Alignment method name.
        datainfo_filename (str): Name of dataset info JSON file.
        output_filename (str): Name of saved alignment matrix file.
        metadata_filename (str): Name of saved metadata JSON file.
        heatmap_filename (str): Name of saved heatmap image.

    Returns:
        dict: Paths and metadata for generated outputs.
    """
    datainfo_path = datainfo_dir / "dataset_info.json"

    alignment_path = find_alignment_matrix_file(align_dir, method=method)
    alignment_matrix = np.load(alignment_path)

    labels, dataset_info = get_labels_from_datainfo(dataset, datainfo_path)
    per_class_alignment = compute_per_class_alignment(alignment_matrix, labels)

    num_classes = per_class_alignment.shape[1]
    class_counts = np.bincount(labels, minlength=num_classes)
    class_proportions = class_counts / class_counts.sum()

    out_dir = output_root / datainfo_dir.name
    out_dir.mkdir(parents=True, exist_ok=True)

    np.save(out_dir / output_filename, per_class_alignment)

    save_alignment_heatmap(
        per_class_alignment=per_class_alignment,
        class_proportions=class_proportions,
        dataset_name=dataset_name,
        save_path=out_dir / heatmap_filename,
        title=f"Per-Class Alignment Heatmap\n{datainfo_dir.name}",
    )

    metadata = {
        "dataset_name": dataset_name,
        "method": method,
        "proportion_key": list(extract_proportion_tuple(datainfo_dir.name)),
        "alignment_dir": str(align_dir),
        "alignment_file_used": str(alignment_path),
        "datainfo_dir": str(datainfo_dir),
        "alignment_shape": list(alignment_matrix.shape),
        "per_class_alignment_shape": list(per_class_alignment.shape),
        "class_counts": class_counts.tolist(),
        "class_proportions": class_proportions.tolist(),
        "indices_count": int(len(dataset_info["indices_D"])),
        "saved_files": {
            "matrix": output_filename,
            "heatmap": heatmap_filename,
            "metadata": metadata_filename,
        },
    }

    with open(out_dir / metadata_filename, "w") as f:
        json.dump(metadata, f, indent=2)

    return {
        "proportion_key": extract_proportion_tuple(datainfo_dir.name),
        "align_dir": align_dir,
        "datainfo_dir": datainfo_dir,
        "alignment_file": alignment_path,
        "output_dir": out_dir,
        "heatmap_path": out_dir / heatmap_filename,
    }


# =========================================================
# DRIVER
# =========================================================
def generate_per_class_alignments(align_root, datainfo_root, output_root, dataset_name, method, selected_proportions=None):
    """
    Run per-class alignment generation across all matched proportion directories.

    Loads dataset, matches alignment/datainfo dirs, computes per-class alignment,
    saves results (matrix, heatmap, metadata), and logs progress.

    Args:
        align_root (Path | str): Root of alignment directories.
        datainfo_root (Path | str): Root of datainfo directories.
        output_root (Path | str): Directory to save outputs.
        dataset_name (str): Dataset identifier.
        method (str): Alignment method filter.
        selected_proportions (list | None): Optional subset of proportions.

    Returns:
        list[dict]: Results for each processed proportion.
    """
    align_root = Path(align_root)
    datainfo_root = Path(datainfo_root)
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    dataset = load_train_dataset(dataset_name)

    matched_pairs = get_matched_pairs(
        align_root=align_root,
        datainfo_root=datainfo_root,
        dataset_name=dataset_name,
        method=method,
        selected_proportions=selected_proportions
    )

    if not matched_pairs:
        raise RuntimeError(
            f"No matching proportion directories found for dataset={dataset_name}, method={method}"
        )

    print(f"Found {len(matched_pairs)} matched pairs for method={method}.\n")

    results = []
    for proportion_key, align_dir, datainfo_dir in matched_pairs:
        print(f"Processing proportion: {proportion_key}")
        result = generate_one_per_class_alignment(
            align_dir=align_dir,
            datainfo_dir=datainfo_dir,
            dataset=dataset,
            output_root=output_root,
            dataset_name=dataset_name,
            method=method,
        )
        results.append(result)
        print(f"  alignment file: {result['alignment_file'].name}")
        print(f"  saved matrix  : {result['output_dir'] / 'per_class_alignment.npy'}")
        print(f"  saved heatmap : {result['heatmap_path']}\n")

    return results

In [ ]:
methods = ["model_baseline","linear","slerp","ties"]
datasets = ["ag_news"]
models = ["bert"]


for model in models:
    for dataset_name in datasets:
        for method in methods:
            
            align_path = f"./results_align_matrix/{model}/{dataset_name}"
            datainfo_path = f"./results_datainfo/{model}/{dataset_name}"
            output_path = f"results_align_matrix_heatmaps/{model}/{dataset_name}/{method}"

            results = generate_per_class_alignments(
                align_root = align_path ,
                datainfo_root = datainfo_path,
                method = method,
                output_root = output_path,
                dataset_name = dataset_name
    )

### Visualizations for linear plots

In [ ]:
# We want the experiments with the class pair where everythign else sums up to a constant number and remins constant. 

In [ ]:
def plot_alignment_vs_proportion_focused(experiments, interpolation, class_pair, class_names, tolerance=0.01):
    """
    Plot how mean alignment scores change as we vary proportions between two classes.
    Focus on experiments where other classes are held at 0.3.
    
    Args:
        experiments: list of experiment dicts
        class_pair: tuple of two class indices to focus on
        class_names: Optional list of class names
        tolerance: tolerance for considering proportions as "fixed at 0.3" (default 0.01)
    """
    class_a, class_b = class_pair
    
    print(f"\n{'='*80}")
    print(f"Analyzing class pair: {class_names[class_a]} (Class {class_a}) vs {class_names[class_b]} (Class {class_b})")
    print(f"{'='*80}")
    
    # Find experiments where the OTHER two classes are both fixed at 0.3
    grouped_experiments = []
    
        # Get the other two class indices (not class_a or class_b)
    other_indices = [i for i in range(len(class_names)) if i not in (class_a, class_b)]
    
    for exp in experiments:
        props = exp['proportions']
        if len(props) < len(class_names):
            continue
        
        # Check if BOTH other classes are approximately 0.3
        other_props = [props[i] for i in other_indices]
        
        consistent = True
        for prop in other_props:
            if(prop!=0.3):
                consistent = False
        
        if(not consistent):
            continue
       
        grouped_experiments.append(exp)
    
    print(f"\nFound {len(grouped_experiments)} experiments where other classes are fixed at 0.3:")
    for exp in grouped_experiments:
        print(f"  {exp['proportions']}")
    
    if len(grouped_experiments) < 2:
        print(f"\nNot enough experiments (need at least 2, found {len(grouped_experiments)})")
        return
    
    # Get the two class indices that are fixed
    other_indices = [i for i in range(len(class_names)) if i not in (class_a, class_b)]
    other_class_names = [class_names[idx] for idx in other_indices]
    
    # Compute mean alignment for each experiment
    data = []
    for exp in grouped_experiments:
        props = exp['proportions']
        alignment_matrix = exp['alignment_matrix']
        labels = exp['labels']
        
        mean_per_class = compute_mean_alignment_per_expert_per_class(
            alignment_matrix, labels
        )
        
        n_experts = alignment_matrix.shape[1]
        
        for expert_idx in range(n_experts):
            for cls in [class_a, class_b]:
                if cls in mean_per_class:
                    data.append({
                        'proportion': props[cls],
                        'class': cls,
                        'expert': expert_idx,
                        'mean_alignment': mean_per_class[cls][expert_idx],
                        'class_a_prop': props[class_a],
                        'class_b_prop': props[class_b],
                        'config_name': f"[{','.join([f'{p:.2f}' for p in props])}]"
                    })
    
    if not data:
        print(f"  No data extracted")
        return
    
    df = pd.DataFrame(data)
    
    # Plot for each expert
    n_experts = df['expert'].nunique()
    n_cols = min(3, n_experts)
    n_rows = (n_experts + n_cols - 1) // n_cols
    
    # Adjust figure size to accommodate title
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 5*n_rows + 1), squeeze=False)
    
    # Build title
    title_text = (f'Mean Alignment vs Class Proportion\n'
                 f'Variable: {class_names[class_a]} & {class_names[class_b]} (sum=0.7) | '
                 f'Fixed: {other_class_names[0]}=0.30'
                 f'Interpolation: {interpolation}')
    
    fig.suptitle(title_text, fontsize=14, fontweight='bold', y=0.98)
    
    for expert_idx in range(n_experts):
        r = expert_idx // n_cols
        c = expert_idx % n_cols
        ax = axes[r, c]
        
        df_expert = df[df['expert'] == expert_idx]
        
        # Get all unique proportions for both classes
        all_props_a = df_expert[df_expert['class'] == class_a]['proportion'].unique()
        all_props_b = df_expert[df_expert['class'] == class_b]['proportion'].unique()
        all_props = np.unique(np.concatenate([all_props_a, all_props_b]))
        
        if len(all_props) > 0:
            min_prop = all_props.min()
            max_prop = all_props.max()
        else:
            min_prop, max_prop = 0, 1
        
        colors = ['#1f77b4', '#ff7f0e']  # Blue and orange
                        
                
        df_cls = df_expert[df_expert['class'] == class_a].sort_values('proportion')
        if len(df_cls) > 0:
            ax.plot(df_cls['proportion'], df_cls['mean_alignment'], 
                    marker='o', label=f'{class_names[class_a]}', linewidth=2.5, 
                    markersize=8, color=colors[0], alpha=0.8)
            
            # Add data point labels
            for _, row in df_cls.iterrows():
                ax.annotate(f'{row["mean_alignment"]:.4f}', 
                            (row['proportion'], row['mean_alignment']),
                            textcoords="offset points", xytext=(0, 10), 
                            ha='center', fontsize=7, alpha=0.7,
                            bbox=dict(boxstyle='round,pad=0.3', facecolor=colors[0], alpha=0.2))


        df_cls = df_expert[df_expert['class'] == class_b].sort_values('proportion')
        df_cls['mean_alignment'] = df_cls['mean_alignment'].to_numpy()[::-1]
        if len(df_cls) > 0:
            ax.plot(df_cls['proportion'], df_cls['mean_alignment'], 
                    marker='o', label=f'{class_names[class_b]}', linewidth=2.5, 
                    markersize=8, color=colors[1], alpha=0.8)
            
            # Add data point labels
            for _, row in df_cls.iterrows():
                ax.annotate(f'{row["mean_alignment"]:.4f}', 
                            (row['proportion'], row['mean_alignment']),
                            textcoords="offset points", xytext=(0, 10), 
                            ha='center', fontsize=7, alpha=0.7,
                            bbox=dict(boxstyle='round,pad=0.3', facecolor=colors[1], alpha=0.2))  
                 
        ax.set_yscale('log')        
        
        # Set x-axis limits with padding
        if max_prop > min_prop:
            padding = (max_prop - min_prop) * 0.15
            ax.set_xlim(min_prop - padding, max_prop + padding)
        
        # Format x-axis to show actual proportion values
        ax.set_xticks(sorted(all_props))
        ax.set_xticklabels([f'{p:.2f}' for p in sorted(all_props)], rotation=0)
        
        ax.axhline(y=0, color='black', linestyle='--', alpha=0.3, linewidth=1)
        ax.set_xlabel(f'Class Proportion (wrt {class_names[class_a]})', fontsize=11, fontweight='bold')
        ax.set_ylabel('Mean Alignment Score', fontsize=11, fontweight='bold')
        ax.set_title(f'Pseudo-Expert {expert_idx + 1}', fontsize=12, fontweight='bold')
        ax.legend(fontsize=10, loc='best', framealpha=0.9)
        ax.grid(True, alpha=0.3)
    
    # Hide unused subplots
    for idx in range(n_experts, n_rows * n_cols):
        r = idx // n_cols
        c = idx % n_cols
        fig.delaxes(axes[r, c])
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])  # Leave space for suptitle
    plt.show()
    
    # Print summary statistics
    print("\n" + "="*80)
    print(f"SUMMARY: Variable Classes: {class_names[class_a]} & {class_names[class_b]}")
    print(f"Fixed: {other_class_names[0]}=0.30")
    print(f"Variable classes share: 0.70 of total proportion")
    print("="*80)
    
    # Group by configuration and print stats
    for config_name in sorted(df['config_name'].unique()):
        df_config = df[df['config_name'] == config_name]
        print(f"\nConfiguration: {config_name}")
        
        for expert_idx in range(n_experts):
            df_expert = df_config[df_config['expert'] == expert_idx]
            if len(df_expert) == 0:
                continue
                
            print(f"  Expert {expert_idx + 1}:")
            
            for cls in [class_a, class_b]:
                df_cls = df_expert[df_expert['class'] == cls]
                if len(df_cls) > 0:
                    prop = df_cls['proportion'].values[0]
                    align = df_cls['mean_alignment'].values[0]
                    print(f"    {class_names[cls]:>15} (prop={prop:.2f}): alignment = {align:9.6f}")
    return data

In [ ]:
def load_experiment_data(dataset_name,interpolation_name,base_dir):
    """
    Load all experiment results with their proportions.
    
    Args:
        base_dir: Path to directory containing experiment folders
        
    Returns:
        list: List of dicts with keys 'proportions', 'alignment_matrix', 'labels', 'output_dir'
    """
    base_path = Path(base_dir)
    
    experiments = []
    
    # Find all directories that match the pattern
    all_dirs = [d for d in base_path.iterdir() if d.is_dir() and d.name.startswith(dataset_name) and interpolation_name in d.name]
    
    print(f"Scanning {len(all_dirs)} directories...")
    
    for output_dir in all_dirs:
        # Try to parse proportions from directory name
        proportions = parse_proportion_from_filename(output_dir.name)
        
        if proportions is None:
            continue
        
        # Check if alignment matrix exists
        alignment_file = f'{output_dir} / alignment_matrix_{interpolation_name}.npy'
        dataset_info_file = output_dir / 'dataset_info.json'
        
        if alignment_file.exists() and dataset_info_file.exists():
            try:
                alignment_matrix = np.load(alignment_file)
                
                with open(dataset_info_file) as f:
                    dataset_info = json.load(f)
                
                experiments.append({
                    'proportions': proportions,
                    'alignment_matrix': alignment_matrix,
                    'labels': dataset_info['dataset'],
                    'output_dir': str(output_dir),
                    'config_name': output_dir.name
                })
                
                # print(f"  ✓ Loaded: {output_dir.name} - shape: {alignment_matrix.shape}, props: {proportions}")
                
            except Exception as e:
                print(f"  ✗ Error loading {output_dir.name}: {e}")
        else:
            missing = []
            if not alignment_file.exists():
                missing.append('alignment_matrix_M.npy')
            if not dataset_info_file.exists():
                missing.append('dataset_info.json')
            print(f"  ⚠ Skipping {output_dir.name}: missing {', '.join(missing)}")
    
    print(f"\n✓ Successfully loaded {len(experiments)} experiments")
    
    # Sort by proportions for easier viewing
    experiments.sort(key=lambda x: tuple(x['proportions']))
    
    return experiments


In [ ]:
import json
import numpy as np
from pathlib import Path
import re

def parse_proportion_from_filename(filename):
    """Extract proportion array from filename like 'ag_news_6e-06_[0.1, 0.3, 0.3, 0.3].json'"""
    match = re.search(r'[\[\(]\s*([0-9eE+\-.,\s]+)\s*[\]\)]', filename)
    if match:
        arr = [float(x) for x in match.group(1).split(",") if x.strip()]
        return arr
    return None

def load_experiment_data(dataset_name, interpolation_name, base_dir):
    """
    Load experiment results with their proportions.
    
    Args:
        dataset_name: Dataset identifier ('snli' or 'ag_news')
        interpolation_name: Method name ('linear', 'slerp', 'ties', 'model_baseline')
        base_dir: Path to results directory
        
    Returns:
        list: Experiment dicts with keys:
              'proportions', 'alignment_matrix', 'labels', 
              'output_dir', 'config_name', 'interpolation'
    """
    base_path = Path(base_dir)
    experiments = []
    
    # Load HuggingFace dataset
    print(f"Loading {dataset_name} dataset from HuggingFace...")
    if dataset_name == 'snli':
        hf_dataset = load_dataset('snli')['train']
    elif dataset_name == 'ag_news':
        hf_dataset = load_dataset('ag_news')['train']
    else:
        raise ValueError(f"Unknown dataset: {dataset_name}")
    
    print(f"Dataset loaded: {len(hf_dataset)} samples")
    
    # Find matching directories
    all_dirs = [d for d in base_path.iterdir() 
                if d.is_dir() and d.name.startswith(dataset_name)]
    
    print(f"Scanning {len(all_dirs)} directories...")
    
    for parent_dir in all_dirs:
        proportions = parse_proportion_from_filename(parent_dir.name)
        
        if proportions is None:
            print(f"  ⚠ Could not parse proportions: {parent_dir.name}")
            continue
        
        # File paths
        dataset_info_file = parent_dir / 'dataset_info.json'
        alignment_subdir = parent_dir / interpolation_name
        alignment_file = alignment_subdir / f'alignment_matrix_{interpolation_name}.npy'
        
        # Check files exist
        if not dataset_info_file.exists():
            print(f"  ⚠ Missing dataset_info.json: {parent_dir.name}")
            continue
        
        if not alignment_file.exists():
            print(f"  ⚠ Missing alignment matrix: {parent_dir.name}/{interpolation_name}")
            continue
        
        # Load data
        with open(dataset_info_file) as f:
            dataset_info = json.load(f)
        
        if 'indices_D' not in dataset_info:
            print(f"  ✗ Missing 'indices_D': {parent_dir.name}")
            continue
        
        indices_D = dataset_info['indices_D']
        labels = [hf_dataset[idx]['label'] for idx in indices_D]
        alignment_matrix = np.load(alignment_file)
        
        # Validate
        if alignment_matrix.shape[0] != len(labels):
            print(f"  ✗ Shape mismatch: {parent_dir.name} "
                  f"({alignment_matrix.shape[0]} vs {len(labels)})")
            continue
        
        experiments.append({
            'proportions': proportions,
            'alignment_matrix': alignment_matrix,
            'labels': labels,
            'output_dir': str(parent_dir),
            'config_name': parent_dir.name,
            'interpolation': interpolation_name
        })
        
        print(f"  ✓ {parent_dir.name}/{interpolation_name} "
              f"({alignment_matrix.shape}, {proportions})")
    
    print(f"\n✓ Loaded {len(experiments)} experiments for {interpolation_name}")
    
    experiments.sort(key=lambda x: tuple(x['proportions']))
    
    return experiments

def compute_mean_alignment_per_expert_per_class(alignment_matrix, class_labels):
    """
    Compute mean alignment score for each pseudo-expert for each class.
    
    Args:
        alignment_matrix: np.ndarray, shape (n_samples, n_pseudo_experts)
        class_labels: 1D array-like of length n_samples with class labels
    
    Returns:
        dict: {class_id: np.array([mean_expert_0, mean_expert_1, ...])}
    """
    class_labels = np.array(class_labels)
    unique_classes = np.unique(class_labels)
    n_pseudo_experts = alignment_matrix.shape[1]
    
    results = {}
    
    for cls in unique_classes:
        cls_mask = (class_labels == cls)
        cls_alignment = alignment_matrix[cls_mask]  # (n_samples_cls, n_experts)
        
        if cls_alignment.shape[0] == 0:
            # No samples for this class
            continue
        
        # Mean across samples for each expert
        mean_per_expert = cls_alignment.mean(axis=0)  # shape (n_experts,)
        results[cls] = mean_per_expert
    
    return results

In [ ]:
def plot_alignment_heatmap_across_proportions(experiments, target_class=0, class_names=None):
    """
    Heatmap showing mean alignment for a target class across all pseudo-experts
    for different proportion configurations.
    
    Args:
        experiments: list of experiment dicts
        target_class: which class to focus on
        class_names: Optional list of class names
    """
    if class_names is None:
        class_names = [f'Class {i}' for i in range(4)]
    
    data = []
    
    for exp in experiments:
        props = exp['proportions']
        alignment_matrix = exp['alignment_matrix']
        labels = exp['labels']
        
        mean_per_class = compute_mean_alignment_per_expert_per_class(
            alignment_matrix, labels
        )
        
        if target_class in mean_per_class:
            # Create readable proportion label
            prop_label = '[' + ', '.join([f'{p:.2f}' for p in props]) + ']'
            
            data.append({
                'proportions': prop_label,
                'target_prop': props[target_class],
                'mean_alignments': mean_per_class[target_class],
                'all_props': props
            })
    
    if len(data) == 0:
        print(f"No data found for class {target_class} ({class_names[target_class]})")
        return
    
    # Sort by target class proportion
    data.sort(key=lambda x: x['target_prop'])
    
    # Create matrix for heatmap
    prop_labels = [d['proportions'] for d in data]
    n_experts = len(data[0]['mean_alignments'])
    
    matrix = np.array([d['mean_alignments'] for d in data])
    
    # Calculate figure height based on number of configurations
    fig_height = max(8, len(data) * 0.5)
    
    fig = plt.figure(figsize=(14, fig_height))
    
    sns.heatmap(matrix, 
                xticklabels=[f'Expert {i+1}' for i in range(n_experts)],
                yticklabels=prop_labels,
                cmap='RdBu_r',
                center=0,
                annot=True,
                fmt='.4f',
                cbar_kws={'label': 'Mean Alignment Score'},
                linewidths=0.5,
                linecolor='gray')
    
    plt.title(f'Mean Alignment Score for {class_names[target_class]} (Class {target_class})\n'
              f'Across Proportion Configurations (Sorted by {class_names[target_class]} proportion)', 
              fontsize=13, fontweight='bold', pad=15)
    plt.xlabel('Pseudo-Expert', fontsize=12, fontweight='bold')
    plt.ylabel('Class Proportions [Class0, Class1, Class2, Class3]', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print("\n" + "="*80)
    print(f"HEATMAP SUMMARY FOR {class_names[target_class]} (Class {target_class})")
    print("="*80)
    print(f"\nConfigurations sorted by {class_names[target_class]} proportion:")
    for d in data:
        print(f"\n  {d['proportions']}")
        print(f"    {class_names[target_class]} proportion: {d['target_prop']:.2f}")
        print(f"    Mean alignment per expert:")
        for i, val in enumerate(d['mean_alignments']):
            print(f"      Expert {i+1}: {val:.6f}")
        print(f"    Overall mean: {np.mean(d['mean_alignments']):9.6f}")
        print(f"    Overall std:  {np.std(d['mean_alignments']):9.6f}")

In [ ]:
def run_experiments(class_names,class_pairs,dataset,interpolation,directory_name):
    
    # Load all experiments
    print("Loading experiments...")
    experiments = load_experiment_data(dataset,interpolation,directory_name)
    print(f"Loaded {len(experiments)} experiments")

    # Show available proportion configurations
    print("\nAvailable proportion configurations:")
    for i, exp in enumerate(experiments):
        print(f"  {i+1}. {exp['proportions']}")

    # Plot alignment vs proportion for all class pairs where other 2 are fixed at 0.3
    print("\n" + "="*80)
    print("PLOTTING ALIGNMENT VS PROPORTION FOR CLASS PAIRS")
    print("="*80)

    for pair in class_pairs:
        print(len(experiments))
        data = plot_alignment_vs_proportion_focused(experiments, interpolation, class_pair=pair, class_names=class_names)
        print(data)

    # Heatmap for each class
    print("\n" + "="*80)
    print("PLOTTING HEATMAPS FOR EACH CLASS")
    print("="*80)

    # for target_class in range(len(class_names)):
    #     plot_alignment_heatmap_across_proportions(experiments, target_class=target_class, class_names=class_names)

In [ ]:
# Define class names for AG News
class_names = ['entailement','neutral','contradiction']

# Plot for class pairs (0,1), (0,2), (0,3), (1,2), (1,3), (2,3)
class_pairs = [(0, 1), (0, 2), (1, 2)]

dataset = 'snli'

interpolations = ['model_baseline','slerp','ties','linear']

directory_name = 'results_align_matrix/results_align_matrix_ag_news'

for interpolation in interpolations: 
    run_experiments(class_names,class_pairs,dataset,interpolation,directory_name)

In [ ]:
def generate_data_for_linear_plots(dataset,interpolation,directory_name,classa,classb):
    
        # Load all experiments
    print("Loading experiments...")
    experiments = load_experiment_data(dataset,interpolation,directory_name)
    print(f"Loaded {len(experiments)} experiments")
    
    
    # Compute mean alignment for each experiment
    data_a = []
    data_b = []
    for exp in experiments:
        props = exp['proportions']
        alignment_matrix = exp['alignment_matrix']
        labels = exp['labels']
        
        mean_per_class = compute_mean_alignment_per_expert_per_class(alignment_matrix, labels)
        
        data_a.append({
            'class_a_proportion': [props[classa]]*15,
            'class_a_alignscores': mean_per_class[classa],
        })
        
        data_b.append({
            'class_b_proportion': [props[classb]]*15,
            'class_b_alignscores': mean_per_class[classb]
        })
    
    return data_a,data_b

In [ ]:
def generate_linear_plots(dataset,interpolation,directory_name,classa,classb):
    
    data_a,data_b = generate_data_for_linear_plots(dataset,interpolation,directory_name,classa,classb)
    
    class_a_prop = np.empty(0)
    class_a_alignscores = np.empty(0)
    for dic in data_a:  
        class_a_prop = np.append(class_a_prop,dic['class_a_proportion'])
        class_a_alignscores = np.append(class_a_alignscores,dic['class_a_alignscores'])
        
    class_b_prop = np.empty(0)
    class_b_alignscores = np.empty(0)
    for dic in data_b:   
        class_b_prop = np.append(class_b_prop,dic['class_b_proportion'])
        class_b_alignscores = np.append(class_b_alignscores,dic['class_b_alignscores'])
        
    plt.scatter(class_a_prop,class_a_alignscores, color='blue', label='Dataset A', marker='o')
    plt.scatter(class_b_prop,class_b_alignscores, color='orange', label='Dataset B', marker='s')
    
    m1, c1, r_value1, p_value1, std_err1 = linregress(class_a_prop,class_a_alignscores)
    plt.plot(class_a_prop, m1*class_a_alignscores + c1, color='blue', linestyle='--', label='Trend A')

    # Trend line for Dataset B
    m2, c2, r_value2, p_value2, std_err2 = linregress(class_b_prop,class_b_alignscores)
    plt.plot(class_b_prop, m2*class_b_alignscores + c2, color='orange', linestyle='-', label='Trend B')

    # 4. Add labels, a legend, and display the plot
    plt.title(f"{interpolation}, (A: {class_names[classa]},B: {class_names[classb]})")
    plt.xlabel("X values")
    plt.ylabel("Y values")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
def generate_box_plots(dataset,interpolation,directory_name,classa,classb):
    data_a,data_b = generate_data_for_linear_plots(dataset,interpolation,directory_name,classa,classb)
    
    # class_a_prop = np.empty(0)
    # class_a_alignscores = np.empty(0)
    # for dic in data_a:  
    #     class_a_prop = np.append(class_a_prop,[dic['class_a_proportion'][0]],axis=0)
    #     class_a_alignscores = np.append(class_a_alignscores,dic['class_a_alignscores'],axis=0)

    # class_b_prop = np.empty(0)
    # class_b_alignscores = np.empty(0)
    # for dic in data_b:   
    #     class_b_prop = np.append(class_b_prop,[dic['class_b_proportion'][0]],axis=0)
    #     class_b_alignscores = np.append(class_b_alignscores,dic['class_b_alignscores'],axis=0)
    
   
    # Prepare data for box plots
    box_data_a = []  # List of arrays, one per unique proportion
    box_data_b = []
    unique_props_a = []
    unique_props_b = []
    
    # Group alignment scores by proportion for class A
    prop_dict_a = {}
    for dic in data_a:
        prop = dic['class_a_proportion'][0]
        scores = dic['class_a_alignscores']
        
        if prop not in prop_dict_a:
            prop_dict_a[prop] = []
        prop_dict_a[prop].extend(scores)
    
    # Convert to sorted lists
    for prop in sorted(prop_dict_a.keys()):
        unique_props_a.append(prop)
        box_data_a.append(prop_dict_a[prop])
    
    # Group alignment scores by proportion for class B
    prop_dict_b = {}
    for dic in data_b:
        prop = dic['class_b_proportion'][0]
        scores = dic['class_b_alignscores']
        
        if prop not in prop_dict_b:
            prop_dict_b[prop] = []
        prop_dict_b[prop].extend(scores)
    
    # Convert to sorted lists
    for prop in sorted(prop_dict_b.keys()):
        unique_props_b.append(prop)
        box_data_b.append(prop_dict_b[prop])
    
    # Create side-by-side box plots
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Box plot for Class A
    bp1 = axes[0].boxplot(box_data_a, 
                          labels=[f'{p:.2f}' for p in unique_props_a],
                          patch_artist=True,
                          widths=0.6)
    
    # Color the boxes
    for patch in bp1['boxes']:
        patch.set_facecolor('#1f77b4')
        patch.set_alpha(0.7)
    
    axes[0].set_xlabel('Class Proportion', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Alignment Score', fontsize=12, fontweight='bold')
    axes[0].set_title(f'{class_names[classa]} - {interpolation}', 
                      fontsize=13, fontweight='bold')
    axes[0].grid(True, alpha=0.3, axis='y')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Box plot for Class B
    bp2 = axes[1].boxplot(box_data_b,
                          labels=[f'{p:.2f}' for p in unique_props_b],
                          patch_artist=True,
                          widths=0.6)
    
    # Color the boxes
    for patch in bp2['boxes']:
        patch.set_facecolor('#ff7f0e')
        patch.set_alpha(0.7)
    
    axes[1].set_xlabel('Class Proportion', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Alignment Score', fontsize=12, fontweight='bold')
    axes[1].set_title(f'{class_names[classb]} - {interpolation}', 
                      fontsize=13, fontweight='bold')
    axes[1].grid(True, alpha=0.3, axis='y')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.suptitle(f'Alignment Score Distribution Across Pseudo-Experts\n'
                 f'{interpolation} | Class Pair: {class_names[classa]} vs {class_names[classb]}',
                 fontsize=14, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"\n{'='*80}")
    print(f"Box Plot Summary: {interpolation}")
    print(f"Class A: {class_names[classa]} | Class B: {class_names[classb]}")
    print(f"{'='*80}")
    
    print(f"\n{class_names[classa]} Statistics:")
    for prop, data in zip(unique_props_a, box_data_a):
        print(f"  Proportion {prop:.2f}: "
              f"median={np.median(data):.6f}, "
              f"mean={np.mean(data):.6f}, "
              f"std={np.std(data):.6f}, "
              f"n={len(data)}")
    
    print(f"\n{class_names[classb]} Statistics:")
    for prop, data in zip(unique_props_b, box_data_b):
        print(f"  Proportion {prop:.2f}: "
              f"median={np.median(data):.6f}, "
              f"mean={np.mean(data):.6f}, "
              f"std={np.std(data):.6f}, "
              f"n={len(data)}")

In [ ]:
for interpolation in interpolations:
    for classa,classb in class_pairs:
        generate_box_plots(dataset,interpolation,directory_name,classa,classb)
        generate_linear_plots(dataset,interpolation,directory_name,classa,classb)

## Visualizations for accuracy distribution

In [ ]:
output_dir = "./accuracy_visualizations"
datainfo_dir = "./results_datainfo"
models = ["gpt2"]
datasets = ["snli"]
file_name = "accuracy_arr.pkl"

In [ ]:
def plot_accuracy_distribution(input_dir, output_dir, file_name):
    """
    Generate accuracy distribution histograms for each training epoch.
    
    Aggregates accuracy data across all experiments and creates
    per-epoch visualizations with mean reference lines.
    
    Args:
        input_dir: Directory containing experiment subdirectories
        output_dir: Directory to save histogram images
        file_name: Name of pickle file containing accuracy arrays
    """
    
    # Setup paths
    base_path = Path(input_dir)
    output_path = Path(output_dir)
    
    # Find all experiment directories
    directories = [d for d in base_path.iterdir() if d.is_dir()]
    
    # Determine number of epochs from first experiment
    with open(f"{directories[0]}/{file_name}", "rb") as f:
        arr = pl.load(f)
        accuracy_arr_size = len(arr)
    
    # Initialize sets to collect unique accuracy values per epoch
    list_of_sets = [set() for i in range(accuracy_arr_size)]
    
    # Aggregate accuracy data across all experiments
    for path in directories:
        with open(f"{path}/{file_name}", "rb") as f:
            arr = pl.load(f)
            for i in range(accuracy_arr_size):
                list_of_sets[i].add(arr[i])
    
    # Create output directory
    output_path.mkdir(parents=True, exist_ok=True)
    
    # ============================================================
    # Generate histogram for each epoch
    # ============================================================
    
    for epoch_idx in range(accuracy_arr_size):
        fig, ax = plt.subplots(figsize=(7, 6))
        
        accuracies = list_of_sets[epoch_idx]
        
        # Plot histogram
        ax.hist(accuracies, bins=20, range=(0, 1),
                color='skyblue', edgecolor='black', alpha=0.7)
        
        ax.set_xlim(0, 1)
        ax.set_xlabel('Accuracy', fontsize=12, fontweight='bold')
        ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
        ax.set_title(f'Checkpoint {epoch_idx + 1}', fontsize=13, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')
        
        # Add mean reference line
        mean_acc = np.mean(list(accuracies))
        ax.axvline(mean_acc, color='red', linestyle='--', linewidth=2,
                   label=f'Mean: {mean_acc:.3f}')
        ax.legend()
        
        plt.suptitle(f'Accuracy Distribution ({len(directories)} experiments)',
                     fontsize=14, fontweight='bold')
        
        plt.tight_layout()
        
        # Save and close
        save_path = output_path / f'accuracy_histogram_{epoch_idx}.png'
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()

In [ ]:
for model_name in models:
    for dataset_name in datasets:
        output_path = f"./{output_dir}/{model_name}/{dataset_name}"
        datainfo_path = f"./{datainfo_dir}/{model_name}/{dataset_name}"
        plot_accuracy_distribution(datainfo_path,output_path,file_name)

## Visualizations for average MAE distribution for the actual generated proportions (from the centroid)

In [ ]:
# Configuration
output_dir = "./centroid_mae_visualizations"
datainfo_dir = "./results_datainfo"
models = ["bert"]
datasets = ["ag_news","yelp_review"]
num_classes_map = {
    "ag_news": 4,
    "yelp_review": 5,
    "snli": 3
}

In [ ]:
# =========================================================
# MAE FROM CENTROID - SIMPLIFIED
# =========================================================

def compute_mae_from_centroid(proportions, num_classes):
    """
    Compute MAE between a proportion vector and the centroid (uniform distribution).
    
    Args:
        proportions: Class proportions that sum to 1
        num_classes: Number of classes
    
    Returns:
        float: Mean absolute error from centroid
    """
    proportions = np.array(proportions)
    centroid = np.full(num_classes, 1.0 / num_classes)
    mae = np.mean(np.abs(proportions - centroid))
    return mae


def plot_mae_from_centroid_distribution(base_dir, dataset_name, num_classes, output_dir=None):
    """
    Generate histogram of MAE from centroid with mean line.
    
    Args:
        base_dir: Directory containing experiment subdirectories
        dataset_name: Dataset name to filter directories
        num_classes: Number of classes
        output_dir: Optional directory to save plot
    """
    base_path = Path(base_dir)
    mae_values = []
    
    # Collect MAE from all experiments
    for exp_dir in base_path.iterdir():
        if not exp_dir.is_dir() or not exp_dir.name.startswith(dataset_name):
            continue
        
        # Parse proportion from directory name
        proportion = extract_proportion_tuple(exp_dir.name)
        
        if proportion is None:
            continue
        
        # Compute MAE from centroid
        mae = compute_mae_from_centroid(proportion, num_classes)
        mae_values.append(mae)
    
    if len(mae_values) == 0:
        print(f"❌ No experiments found for {dataset_name}")
        return
    
    # Create histogram
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.hist(mae_values, bins=30, color='skyblue', edgecolor='black', alpha=0.7)
    
    # Add mean line
    mean_mae = np.mean(mae_values)
    ax.axvline(mean_mae, color='red', linestyle='--', linewidth=2,
               label=f'Mean: {mean_mae:.4f}')
    
    ax.set_xlabel('MAE from Centroid', fontsize=12, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
    ax.set_title(f'MAE from Centroid Distribution - {dataset_name}\n'
                 f'({len(mae_values)} experiments)', 
                 fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    
    # Save if output directory specified
    if output_dir:
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)
        save_path = output_path / f'mae_from_centroid_{dataset_name}.png'
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"✓ Saved: {save_path}")
    
    plt.show()
    
    print(f"\nTotal experiments: {len(mae_values)}")
    print(f"Mean MAE: {mean_mae:.6f}")

In [ ]:
# Generate plots
for model in models:
    for dataset in datasets:
        plot_mae_from_centroid_distribution(
            base_dir=f"{datainfo_dir}/{model}/{dataset}",
            dataset_name=dataset,
            num_classes=num_classes_map[dataset],
            output_dir=f"{output_dir}/{model}"
        )

## Visualization for alignment matrices

In [ ]:
# =========================================================
# RAW ALIGNMENT MATRIX HEATMAP - SIMPLE
# =========================================================

def plot_raw_alignment_heatmap(alignment_matrix, labels, dataset_name, proportion_tuple):
    """
    Generate heatmap of raw alignment matrix.
    
    Args:
        alignment_matrix: np.ndarray, shape (n_samples, n_experts)
        labels: array of class labels for each sample
        dataset_name: Name of dataset
        proportion_tuple: Tuple of class proportions
    """
    # Sort samples by class
    sort_idx = np.argsort(labels)
    sorted_matrix = alignment_matrix[sort_idx]
    sorted_labels = labels[sort_idx]
    
    num_classes = int(labels.max()) + 1
    n_experts = alignment_matrix.shape[1]
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Plot heatmap
    im = ax.imshow(sorted_matrix, aspect='auto', cmap='RdYlGn', 
                   vmin=-3.5, vmax=0)
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Alignment Score', fontsize=12, fontweight='bold')
    
    # Add class boundary lines
    current_pos = 0
    for c in range(num_classes):
        count = np.sum(sorted_labels == c)
        if count > 0:
            if current_pos + count < len(sorted_labels):
                ax.axhline(y=current_pos + count - 0.5, 
                          color='white', linewidth=2, linestyle='--')
            current_pos += count
    
    # Labels
    ax.set_xlabel('Pseudo-Expert', fontsize=12, fontweight='bold')
    ax.set_ylabel('Samples (grouped by class)', fontsize=12, fontweight='bold')
    ax.set_xticks(range(n_experts))
    ax.set_xticklabels([f'θ_{i}' for i in range(n_experts)])
    
    # Title
    prop_str = '[' + ', '.join([f'{p:.2f}' for p in proportion_tuple]) + ']'
    plt.title(f'Alignment Matrix - {dataset_name}\nProportions: {prop_str}',
              fontsize=14, fontweight='bold', pad=20)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Load one experiment
dataset_name = "ag_news"
method = "linear"

# Load dataset
dataset = load_train_dataset(dataset_name)

# Pick one experiment directory
exp_dir = Path("/home/aditya/hack_model/results_datainfo/bert/ag_news/ag_news_(0.0, 0.45, 0.09, 0.46)")
proportion = extract_proportion_tuple(exp_dir.name)

# Load labels
with open(exp_dir / 'dataset_info.json') as f:
    dataset_info = json.load(f)
indices = dataset_info['indices_D']
labels = np.array([dataset[idx]['label'] for idx in indices])

# Load alignment matrix
align_dir = Path("/home/aditya/hack_model/results_align_matrix/bert/ag_news/ag_news_linear_[0.0, 0.45, 0.09, 0.46]")
alignment_matrix = np.load(align_dir / f'alignment_matrix_{method}_max.npy')

# Plot
plot_raw_alignment_heatmap(alignment_matrix, labels, dataset_name, proportion)